# ⚙️ ML Feature Pipeline — Lakeflow Spark Declarative Pipeline
Reactive feature computation pipeline. When new data lands in bronze tables, this pipeline recomputes silver features and assembles the gold prediction feature vector.

**Pipeline Flow:**
```
bronze.news_rss_all ─────────────────┐
bronze.fred_macro_indicators ────────┤
silver.stock_prices ────────────────┤───▶ gold.ml_prediction_features
bronze.historical_news_gdelt ────────┤
silver.forecast_features_daily ──────┘
```

In [0]:
# ── Pipeline Imports ──────────────────────────────────────────
import dlt
from pyspark.sql import functions as F, Window

CATALOG = "riskbricks"

FOCUS_SYMBOLS = [
    "LMT", "RTX", "NOC", "GD", "BA", "HII",
    "XOM", "CVX", "COP", "SLB", "HAL", "OXY",
    "JPM", "BAC", "GS", "MS", "C", "WFC",
    "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA",
    "INTC", "AMD", "AVGO", "QCOM", "MU", "LRCX", "AMAT",
    "WMT", "COST", "HD", "NKE", "MCD", "SBUX",
    "JNJ", "PFE", "UNH", "LLY", "ABBV", "MRK",
    "CAT", "DE", "HON", "GE", "MMM",
    "UAL", "DAL", "AAL",
]

SECTOR_MAP = {
    "LMT":"Defense","RTX":"Defense","NOC":"Defense","GD":"Defense","BA":"Defense","HII":"Defense",
    "XOM":"Energy","CVX":"Energy","COP":"Energy","SLB":"Energy","HAL":"Energy","OXY":"Energy",
    "JPM":"Banks","BAC":"Banks","GS":"Banks","MS":"Banks","C":"Banks","WFC":"Banks",
    "AAPL":"BigTech","MSFT":"BigTech","GOOGL":"BigTech","AMZN":"BigTech","NVDA":"BigTech","META":"BigTech","TSLA":"BigTech",
    "INTC":"Semis","AMD":"Semis","AVGO":"Semis","QCOM":"Semis","MU":"Semis","LRCX":"Semis","AMAT":"Semis",
    "WMT":"Consumer","COST":"Consumer","HD":"Consumer","NKE":"Consumer","MCD":"Consumer","SBUX":"Consumer",
    "JNJ":"Pharma","PFE":"Pharma","UNH":"Pharma","LLY":"Pharma","ABBV":"Pharma","MRK":"Pharma",
    "CAT":"Industrial","DE":"Industrial","HON":"Industrial","GE":"Industrial","MMM":"Industrial",
    "UAL":"Airlines","DAL":"Airlines","AAL":"Airlines",
}
sym_list = ",".join([f"'{s}'" for s in FOCUS_SYMBOLS])

In [0]:
@dlt.table(
    name="technical_indicators",
    comment="RSI(14), MACD histogram, Bollinger Band %, volume ratio, overnight gap, close position",
    table_properties={"quality": "silver"},
)
def technical_indicators():
    w = Window.partitionBy("symbol").orderBy("date")
    w14 = Window.partitionBy("symbol").orderBy("date").rowsBetween(-13, 0)
    w20 = Window.partitionBy("symbol").orderBy("date").rowsBetween(-19, 0)
    w12 = Window.partitionBy("symbol").orderBy("date").rowsBetween(-11, 0)
    w26 = Window.partitionBy("symbol").orderBy("date").rowsBetween(-25, 0)
    w9  = Window.partitionBy("symbol").orderBy("date").rowsBetween(-8, 0)
    w5  = Window.partitionBy("symbol").orderBy("date").rowsBetween(-4, 0)

    prices = spark.sql(f"""
        SELECT symbol, DATE(date) AS date, open, high, low, close, volume
        FROM {CATALOG}.silver.stock_prices
        WHERE symbol IN ({sym_list}) AND DATE(date) >= DATE_SUB(CURRENT_DATE(), 60)
    """)

    tech = (prices
        .withColumn("prev_close", F.lag("close").over(w))
        .withColumn("change", F.col("close") - F.col("prev_close"))
        .withColumn("gain", F.when(F.col("change") > 0, F.col("change")).otherwise(0))
        .withColumn("loss", F.when(F.col("change") < 0, F.abs(F.col("change"))).otherwise(0))
        .withColumn("avg_gain_14", F.avg("gain").over(w14))
        .withColumn("avg_loss_14", F.avg("loss").over(w14))
        .withColumn("rsi_14", 100 - (100 / (1 + F.col("avg_gain_14") / F.greatest(F.col("avg_loss_14"), F.lit(0.0001)))))
        .withColumn("ema_12", F.avg("close").over(w12))
        .withColumn("ema_26", F.avg("close").over(w26))
        .withColumn("macd_line", F.col("ema_12") - F.col("ema_26"))
        .withColumn("macd_signal", F.avg("macd_line").over(w9))
        .withColumn("macd_hist", F.col("macd_line") - F.col("macd_signal"))
        .withColumn("bb_mid", F.avg("close").over(w20))
        .withColumn("bb_std", F.stddev("close").over(w20))
        .withColumn("bb_pct", (F.col("close") - (F.col("bb_mid") - 2*F.col("bb_std"))) / F.greatest((4*F.col("bb_std")), F.lit(0.01)))
        .withColumn("avg_vol_20", F.avg("volume").over(w20))
        .withColumn("vol_ratio", F.col("volume") / F.greatest(F.col("avg_vol_20"), F.lit(1)))
        .withColumn("gap_pct", (F.col("open") - F.col("prev_close")) / F.col("prev_close"))
        .withColumn("close_position", (F.col("close") - F.col("low")) / F.greatest(F.col("high") - F.col("low"), F.lit(0.01)))
    )

    return tech.select("symbol", "date",
        F.round("rsi_14",2).alias("rsi_14"), F.round("macd_hist",4).alias("macd_hist"),
        F.round("bb_pct",3).alias("bb_pct"), F.round("vol_ratio",2).alias("vol_ratio"),
        F.round("gap_pct",4).alias("gap_pct"), F.round("close_position",3).alias("close_position"))

In [0]:
@dlt.table(
    name="sector_features",
    comment="Sector-relative momentum, sector breadth for 52 focus stocks across 9 sectors",
    table_properties={"quality": "silver"},
)
def sector_features():
    sector_rows = [(k, v) for k, v in SECTOR_MAP.items()]
    sector_df = spark.createDataFrame(sector_rows, ["symbol", "sector"])

    prices_daily = spark.sql(f"""
        SELECT symbol, DATE(date) AS date, close,
               (close - LAG(close) OVER (PARTITION BY symbol ORDER BY date)) /
               LAG(close) OVER (PARTITION BY symbol ORDER BY date) AS daily_return
        FROM {CATALOG}.silver.stock_prices
        WHERE symbol IN ({sym_list}) AND DATE(date) >= DATE_SUB(CURRENT_DATE(), 30)
    """)

    ps = prices_daily.join(sector_df, "symbol")
    sector_avg = ps.groupBy("sector", "date").agg(
        F.avg("daily_return").alias("sector_avg_return"),
        F.count("*").alias("sector_count"),
        F.sum(F.when(F.col("daily_return") > 0, 1).otherwise(0)).alias("sector_up"))

    svs = ps.join(sector_avg, ["sector", "date"]) \
        .withColumn("stock_vs_sector", F.col("daily_return") - F.col("sector_avg_return")) \
        .withColumn("sector_breadth", F.col("sector_up") / F.col("sector_count"))

    w5s = Window.partitionBy("symbol").orderBy("date").rowsBetween(-4, 0)
    w5sec = Window.partitionBy("sector").orderBy("date").rowsBetween(-4, 0)
    svs = svs.withColumn("sector_rel_5d", F.sum("stock_vs_sector").over(w5s)) \
        .withColumn("sector_momentum_5d", F.sum("sector_avg_return").over(w5sec))

    return svs.select("symbol", "date", "sector",
        F.round("sector_rel_5d",4).alias("sector_rel_5d"),
        F.round("sector_momentum_5d",4).alias("sector_momentum_5d"),
        F.round("sector_breadth",3).alias("sector_breadth"),
        F.round("stock_vs_sector",4).alias("stock_vs_sector_1d"))

In [0]:
@dlt.table(
    name="market_breadth",
    comment="Daily advance/decline ratio and % above 20-day MA from full stock universe",
    table_properties={"quality": "silver"},
)
def market_breadth():
    return spark.sql(f"""
        WITH daily AS (
            SELECT DATE(date) AS date, symbol, close,
                   AVG(close) OVER (PARTITION BY symbol ORDER BY date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS ma_20,
                   (close - LAG(close) OVER (PARTITION BY symbol ORDER BY date)) / LAG(close) OVER (PARTITION BY symbol ORDER BY date) AS ret
            FROM {CATALOG}.silver.stock_prices WHERE DATE(date) >= DATE_SUB(CURRENT_DATE(), 30)
        )
        SELECT date, AVG(ret) AS market_return,
               SUM(CASE WHEN ret > 0 THEN 1 ELSE 0 END) / COUNT(*) AS advance_ratio,
               SUM(CASE WHEN close > ma_20 THEN 1 ELSE 0 END) / COUNT(*) AS pct_above_ma20,
               STDDEV(ret) AS market_dispersion
        FROM daily WHERE ret IS NOT NULL GROUP BY date
    """)

In [0]:
@dlt.view(
    name="news_ai_sentiment_latest",
    comment="AI-classified sentiment from last 3 days of RSS headlines per symbol",
)
def news_ai_sentiment_latest():
    return spark.sql(f"""
        WITH classified AS (
            SELECT symbol,
                   ai_classify(title, ARRAY('very_positive','positive','neutral','negative','very_negative')) AS sent
            FROM {CATALOG}.bronze.news_rss_all
            WHERE symbol IN ({sym_list})
              AND published_date >= DATE_SUB(CURRENT_DATE(), 3)
              AND title IS NOT NULL AND LENGTH(title) > 10
        )
        SELECT symbol,
               AVG(CASE sent WHEN 'very_positive' THEN 2.0 WHEN 'positive' THEN 1.0
                    WHEN 'neutral' THEN 0.0 WHEN 'negative' THEN -1.0 WHEN 'very_negative' THEN -2.0 END) AS ai_sentiment,
               COUNT(*) AS news_count
        FROM classified GROUP BY symbol
    """)

In [0]:
@dlt.table(
    name="ml_prediction_features",
    comment="Final 17-feature vector for ML prediction — joins all silver sources",
    table_properties={"quality": "gold"},
)
def ml_prediction_features():
    return spark.sql(f"""
        WITH prices AS (
            SELECT symbol, last_close, return_5d, return_20d, volatility_20d, as_of_date
            FROM {CATALOG}.silver.forecast_features_daily
            WHERE symbol IN ({sym_list})
              AND as_of_date = (SELECT MAX(as_of_date) FROM {CATALOG}.silver.forecast_features_daily)
        ),
        tech AS (
            SELECT * FROM LIVE.technical_indicators
            WHERE date = (SELECT MAX(date) FROM LIVE.technical_indicators)
        ),
        sector AS (
            SELECT * FROM LIVE.sector_features
            WHERE date = (SELECT MAX(date) FROM LIVE.sector_features)
        ),
        breadth AS (
            SELECT * FROM LIVE.market_breadth
            WHERE date = (SELECT MAX(date) FROM LIVE.market_breadth)
        ),
        macro AS (
            SELECT indicator, value FROM {CATALOG}.bronze.fred_macro_indicators
            WHERE date = (SELECT MAX(date) FROM {CATALOG}.bronze.fred_macro_indicators WHERE indicator = 'VIX')
        ),
        ai AS (SELECT * FROM LIVE.news_ai_sentiment_latest),
        gdelt AS (
            SELECT symbol, AVG(avg_tone) AS gdelt_tone, COUNT(*) AS gdelt_events
            FROM {CATALOG}.bronze.historical_news_gdelt
            WHERE event_date >= DATE_SUB(CURRENT_DATE(), 5) AND symbol IN ({sym_list})
            GROUP BY symbol
        )
        SELECT p.symbol, p.as_of_date AS pred_date, p.last_close,
               p.return_5d, p.return_20d, p.volatility_20d,
               COALESCE(a.ai_sentiment, 0) AS ai_sentiment,
               COALESCE(a.news_count, 0) AS news_count,
               COALESCE(g.gdelt_tone, 0) AS gdelt_tone,
               COALESCE(g.gdelt_events, 0) AS gdelt_events,
               COALESCE(t.rsi_14, 50) AS rsi_14,
               COALESCE(t.macd_hist, 0) AS macd_hist,
               COALESCE(t.gap_pct, 0) AS gap_pct,
               COALESCE(s.sector_momentum_5d, 0) AS sector_momentum_5d,
               COALESCE(s.sector_breadth, 0.5) AS sector_breadth,
               COALESCE(b.advance_ratio, 0.5) AS advance_ratio,
               COALESCE(b.pct_above_ma20, 0.5) AS pct_above_ma20,
               COALESCE(MAX(CASE WHEN m.indicator='VIX' THEN m.value END), 20) AS vix,
               30 AS days_to_earnings,
               CASE WHEN DAYOFWEEK(p.as_of_date) = 2 THEN 1 ELSE 0 END AS is_monday
        FROM prices p
        LEFT JOIN ai a ON p.symbol = a.symbol
        LEFT JOIN gdelt g ON p.symbol = g.symbol
        LEFT JOIN tech t ON p.symbol = t.symbol
        LEFT JOIN sector s ON p.symbol = s.symbol
        CROSS JOIN breadth b
        CROSS JOIN macro m
        GROUP BY p.symbol, p.as_of_date, p.last_close, p.return_5d, p.return_20d, p.volatility_20d,
                 a.ai_sentiment, a.news_count, g.gdelt_tone, g.gdelt_events,
                 t.rsi_14, t.macd_hist, t.gap_pct, s.sector_momentum_5d, s.sector_breadth,
                 b.advance_ratio, b.pct_above_ma20, p.as_of_date
    """)